
# Medición de rendimiento con `time.perf_counter()` en AppTickets

`time.perf_counter()` se utiliza para medir tiempos de ejecución en una aplicación orientada a objetos basada en tickets.

La intención no es “optimizar por optimizar”, sino aprender a responder preguntas como:

- ¿Cuánto tarda crear varios tickets?
- ¿Cuánto tarda validar muchos folios?
- ¿Cuánto tarda atender tickets?
- ¿Qué pasa cuando un método también escribe en un archivo de log?
- ¿Cómo comparar dos bloques de código de forma clara?

> `time.perf_counter()` es ideal para medir intervalos cortos de tiempo porque usa un contador de alta resolución.  
> Su valor absoluto no importa; lo importante es la diferencia entre el tiempo inicial y el tiempo final.



## 1. Importar módulos necesarios

Usaremos:

- `time`: para medir duración con `perf_counter()`.
- `os`: para revisar o eliminar el archivo de log.
- `datetime`, `re`, `ABC`, `abstractmethod` y `wraps`: porque forman parte de la aplicación de tickets.


In [ ]:

from abc import ABC, abstractmethod
from functools import wraps
from datetime import datetime
import re
import time
import os
import pandas as pd



## 2. Aplicación base: AppTickets

La siguiente celda concentra la aplicación de tickets para que el cuaderno pueda ejecutarse de forma independiente.

La aplicación incluye:

- Clase abstracta `TicketBase`.
- Clase principal `Ticket`.
- Clases especializadas:
  - `TicketSoporte`
  - `TicketDesarrollo`
  - `TicketInfraestructura`
- Encapsulación con propiedades.
- Validación de folios con `@staticmethod`.
- Contador global con `@classmethod`.
- Decorador `registrar_accion` para guardar historial y escribir en archivo de log.

> Nota didáctica: como algunos métodos escriben en `registro_acciones.log`, las mediciones de esos métodos incluyen el costo de escritura a archivo.


In [ ]:
# ============================================================================
# IMPORTACIONES NECESARIAS
# ============================================================================
from abc import ABC, abstractmethod  # Para crear clases abstractas
from functools import wraps  # Para preservar metadatos en decoradores
from datetime import datetime  # Para obtener fecha y hora actual
import re  # Para validar formato de folio con expresiones regulares

# ============================================================================
# DECORADOR: registrar_accion
# ============================================================================
# Este decorador registra automáticamente todas las acciones realizadas
# sobre los tickets (cambios de estado, cierre, etc.)
# ============================================================================

def registrar_accion(funcion):
    """
    Decorador que envuelve métodos para registrar sus acciones.
    
    Captura:
    - Fecha y hora de la acción
    - Clase y método que se ejecutó
    - Folio del ticket
    - Estado anterior y posterior
    
    Guarda el registro en un archivo .log y en el historial del objeto
    """
    @wraps(funcion)  # Preserva el nombre y docstring de la función original
    def envoltura(*args, **kwargs):
        # args[0] es el objeto (self) que llamó al método
        objeto = args[0]

        # Captura la fecha y hora actual en formato legible
        fecha_hora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Obtiene el nombre de la clase del objeto
        clase = objeto.__class__.__name__
        
        # Obtiene el nombre del método que se está decorando
        metodo = funcion.__name__
        
        # Obtiene el folio del objeto, o "N/A" si no existe
        folio = getattr(objeto, "folio", "N/A")
        
        # Captura el estado ANTES de ejecutar la acción
        estado_anterior = getattr(objeto, "estado", "N/A")

        # EJECUTA la función original (método decorado)
        resultado = funcion(*args, **kwargs)

        # Captura el estado DESPUÉS de ejecutar la acción
        estado_nuevo = getattr(objeto, "estado", "N/A")

        # Crea un diccionario con toda la información del evento
        registro = {
            "fecha_hora": fecha_hora,
            "clase": clase,
            "folio": folio,
            "metodo": metodo,
            "estado_anterior": estado_anterior,
            "estado_nuevo": estado_nuevo
        }

        # Si el objeto tiene historial, añade este registro
        if hasattr(objeto, "_historial"):
            objeto._historial.append(registro)

        # Escribe el registro en un archivo de log para auditoría
        with open("registro_acciones.log", "a", encoding="utf-8") as archivo:
            archivo.write(
                f"{fecha_hora} | "
                f"{clase} | "
                f"{folio} | "
                f"{metodo} | "
                f"{estado_anterior} -> {estado_nuevo}\n"
            )

        # Retorna el resultado de la función original
        return resultado

    return envoltura


# ============================================================================
# CLASE ABSTRACTA: TicketBase
# ============================================================================
# Define la interfaz que deben cumplir todas las clases Ticket
# ============================================================================

class TicketBase(ABC):
    """
    Clase abstracta base para todos los tipos de tickets.
    
    Define el contrato que DEBEN implementar sus subclases.
    Garantiza que todo ticket tenga un método atender().
    """

    @abstractmethod  # Obliga a las subclases a implementar este método
    def atender(self):
        """Método que debe ser implementado por todas las subclases"""
        pass


# ============================================================================
# CLASE PRINCIPAL: Ticket
# ============================================================================
# Implementa la funcionalidad base de un ticket de soporte
# ============================================================================

class Ticket(TicketBase):
    """
    Representa un ticket de soporte genérico.
    
    Atributos:
    - folio: identificador único (ej: TK-123)
    - titulo: asunto del ticket
    - descripcion: detalles del problema
    - estado: situación actual (Abierto, En proceso, Cerrado)
    """

    # Lista de estados válidos que puede tener un ticket
    ESTADOS_VALIDOS = ["Abierto", "En proceso", "Cerrado"]
    
    # Contador de clase para saber cuántos tickets se han creado
    _contador_global = 0

    def __init__(self, folio, titulo, descripcion):
        """
        Inicializa un nuevo ticket.
        
        Args:
            folio: identificador (se valida automáticamente)
            titulo: asunto del ticket
            descripcion: descripción del problema
        """
        # Crea lista vacía para guardar el historial de cambios
        self._historial = []

        # Establece los atributos (usan los setters para validación)
        self.folio = folio
        self.titulo = titulo
        self.descripcion = descripcion
        self.estado = "Abierto"  # Todo ticket inicia en estado Abierto

        # Incrementa el contador global de tickets
        Ticket._contador_global += 1

        # Registra manualmente el evento de creación
        self._registrar_evento_manual(
            "crear",
            "N/A",  # No hay estado anterior
            self.estado
        )

    # ========================================================================
    # PROPIEDADES (Property): Getters y Setters con validación
    # ========================================================================
    # Las propiedades permiten usar notación punto (obj.folio) pero
    # internamente llamar métodos para validar los datos
    # ========================================================================

    @property
    def folio(self):
        """Getter: devuelve el folio del ticket"""
        return self._folio

    @folio.setter
    def folio(self, nuevo_folio):
        """
        Setter: valida y establece el folio.
        
        Formato válido: TK-123 o TK123
        Lanza ValueError si el formato no es válido
        """
        if not self.es_folio_valido(nuevo_folio):
            raise ValueError(f"Folio no válido: {nuevo_folio}")
        self._folio = nuevo_folio

    @property
    def titulo(self):
        """Getter: devuelve el título del ticket"""
        return self._titulo

    @titulo.setter
    def titulo(self, nuevo_titulo):
        """Setter: establece el título sin validación especial"""
        self._titulo = nuevo_titulo

    @property
    def descripcion(self):
        """Getter: devuelve la descripción del ticket"""
        return self._descripcion

    @descripcion.setter
    def descripcion(self, nueva_descripcion):
        """Setter: establece la descripción sin validación especial"""
        self._descripcion = nueva_descripcion

    @property
    def estado(self):
        """Getter: devuelve el estado actual del ticket"""
        return self._estado

    @estado.setter
    def estado(self, nuevo_estado):
        """
        Setter: valida que el estado sea válido.
        
        Solo permite: "Abierto", "En proceso", "Cerrado"
        Lanza ValueError si el estado no es válido
        """
        if nuevo_estado not in self.ESTADOS_VALIDOS:
            raise ValueError(f"Estado no válido: {nuevo_estado}")
        self._estado = nuevo_estado

    @property
    def historial(self):
        """
        Getter: devuelve el historial como tupla (inmutable).
        
        Devuelve tupla en lugar de lista para evitar
        que se modifique desde fuera
        """
        return tuple(self._historial)

    # ========================================================================
    # MÉTODOS PRIVADOS
    # ========================================================================

    def _registrar_evento_manual(self, metodo, estado_anterior, estado_nuevo):
        """
        Registra manualmente un evento en el historial.
        
        Se usa para eventos que NO están decorados con @registrar_accion
        (como la creación inicial del ticket)
        """
        registro = {
            "fecha_hora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "clase": self.__class__.__name__,
            "folio": self.folio,
            "metodo": metodo,
            "estado_anterior": estado_anterior,
            "estado_nuevo": estado_nuevo
        }
        self._historial.append(registro)

    # ========================================================================
    # MÉTODOS PÚBLICOS (decorados con @registrar_accion)
    # ========================================================================

    @registrar_accion  # Automáticamente registra la acción
    def cambiar_estado(self, nuevo_estado):
        """Cambia el estado del ticket al especificado"""
        self.estado = nuevo_estado

    @registrar_accion  # Automáticamente registra la acción
    def cerrar(self):
        """Cierra el ticket (cambia estado a 'Cerrado')"""
        self.estado = "Cerrado"

    # ========================================================================
    # MÉTODOS DE CLASE
    # ========================================================================

    @classmethod
    def total_tickets_creados(cls):
        """
        Devuelve el total de tickets creados.
        
        Es método de clase porque usa _contador_global de la clase,
        no de instancias individuales
        """
        return cls._contador_global

    # ========================================================================
    # MÉTODOS ESTÁTICOS
    # ========================================================================

    @staticmethod
    def es_folio_valido(folio):
        """
        Valida el formato del folio usando expresión regular.
        
        Formato válido: TK-123 o TK123
        
        Explicación de la regex:
        ^       = inicio de la cadena
        TK      = literal "TK"
        -?      = guión opcional (0 o 1)
        \\d{3}  = exactamente 3 dígitos
        $       = fin de la cadena
        
        Returns:
            True si el folio es válido, False en caso contrario
        """
        return bool(re.fullmatch(r"^TK-?\d{3}$", folio))

    # ========================================================================
    # MÉTODOS DE INSTANCIA - LÓGICA DE NEGOCIO
    # ========================================================================

    def registrar_atencion(self):
        """
        Marca el ticket como "En proceso".
        
        NO está decorado porque es un paso previo,
        la decoración ocurre en atender()
        """
        self.estado = "En proceso"
        return f"Ticket {self.folio} marcado como En proceso."

    @registrar_accion  # Registra automáticamente esta acción
    def atender(self):
        """
        Atiende el ticket (lo marca como En proceso).
        
        Implementa el método abstracto de TicketBase
        """
        return self.registrar_atencion()

    # ========================================================================
    # MÉTODO ESPECIAL: __str__
    # ========================================================================

    def __str__(self):
        """
        Representación en string del ticket.
        
        Se usa cuando se imprime: print(ticket)
        """
        return (
            f"{self.__class__.__name__} | "
            f"Folio: {self.folio} | "
            f"Título: {self.titulo} | "
            f"Estado: {self.estado}"
        )


# ============================================================================
# CLASES ESPECIALIZADAS
# ============================================================================
# Heredan de Ticket pero personalizan el método atender()
# para diferentes tipos de soporte
# ============================================================================

class TicketSoporte(Ticket):
    """
    Ticket especializado para soporte técnico.
    
    Cuando se atiende, muestra un mensaje indicando que será
    atendido por la mesa de ayuda.
    """

    @registrar_accion  # Registra automáticamente
    def atender(self):
        """
        Atiende el ticket de soporte.
        
        Llama al método atender() de la clase padre y añade
        un mensaje especifico para soporte.
        """
        mensaje_base = super().atender()  # Ejecuta el atender() de Ticket
        return (
            f"{mensaje_base} "
            f"El ticket de soporte {self.folio} está siendo atendido por mesa de ayuda."
        )


class TicketDesarrollo(Ticket):
    """
    Ticket especializado para desarrollo de software.
    
    Cuando se atiende, muestra un mensaje indicando que fue
    asignado al equipo de programación.
    """

    @registrar_accion  # Registra automáticamente
    def atender(self):
        """
        Atiende el ticket de desarrollo.
        
        Llama al método atender() de la clase padre y añade
        un mensaje especifico para desarrollo.
        """
        mensaje_base = super().atender()  # Ejecuta el atender() de Ticket
        return (
            f"{mensaje_base} "
            f"El ticket de desarrollo {self.folio} fue asignado al equipo de programación."
        )


class TicketInfraestructura(Ticket):
    """
    Ticket especializado para infraestructura.
    
    Cuando se atiende, muestra un mensaje indicando que fue
    enviado al área de servidores y redes.
    """

    @registrar_accion  # Registra automáticamente
    def atender(self):
        """
        Atiende el ticket de infraestructura.
        
        Llama al método atender() de la clase padre y añade
        un mensaje especifico para infraestructura.
        """
        mensaje_base = super().atender()  # Ejecuta el atender() de Ticket
        return (
            f"{mensaje_base} "
            f"El ticket de infraestructura {self.folio} fue enviado al área de servidores/redes."
        )



## 3. Primera medición simple con `perf_counter()`

La estructura básica es:

```python
inicio = time.perf_counter()
# código que queremos medir
fin = time.perf_counter()

duracion = fin - inicio
```

La diferencia `fin - inicio` representa el tiempo transcurrido en segundos.


In [ ]:
# ============================================================================



## 4. Función auxiliar para medir bloques de código

Para no repetir la misma estructura muchas veces, crearemos una función llamada `medir_tiempo`.

Esta función recibe:

- una descripción;
- una función a ejecutar;
- argumentos opcionales para esa función.
- *args y **kwargs permiten pasar cualquier número de argumentos posicionales y de palabra clave a la función que se está midiendo. Esto hace que el decorador sea flexible y pueda medir el tiempo de cualquier función, sin importar su firma.

Devuelve un diccionario con el resultado de la medición.


In [ ]:
#



## 5. Medir validación de folios

La validación de folios es una operación pequeña, pero es buena para demostrar una idea importante:

> Cuando una operación es muy rápida, conviene ejecutarla muchas veces para obtener una medición más visible.

En este caso validaremos 900 folios correctos y 900 folios incorrectos.


In [ ]:
#



## 6. Crear varios tickets y medir el tiempo

Ahora mediremos la creación de varios objetos.

Como el patrón de folio solo permite tres dígitos, usaremos como máximo 900 tickets para mantener folios válidos como `TK-001`, `TK-002`, ..., `TK-900`.


In [ ]:
#



## 7. Medir atención de tickets

En esta aplicación, atender un ticket no solo cambia su estado. También pasa por el decorador `registrar_accion`.

Eso significa que al ejecutar `ticket.atender()` ocurren varias cosas:

1. Se consulta el estado anterior.
2. Se ejecuta la atención.
3. Se cambia el estado a `En proceso`.
4. Se agrega un registro al historial interno.
5. Se escribe una línea en `registro_acciones.log`.

Por eso esta medición no representa únicamente “cambiar un atributo”, sino una operación de negocio más completa.


In [ ]:
#



## 8. Medir cierre de tickets

Cerraremos solo los primeros 100 tickets para observar otra operación decorada.

La operación `cerrar()` también registra historial y escribe en archivo de log.

### Slicing 
es el proceso de obtener una porción de una lista, tupla o cadena. En este caso, se está obteniendo los primeros 100 tickets de la lista completa de tickets.
La sintaxis general para slicing es: 
``` text
lista[inicio:fin:paso]
```
donde inicio es el índice inicial (inclusive), fin es el índice final (exclusive) y paso es la cantidad de elementos que se saltan. 
Es decir, obtiene una sublista que comienza con el elemento en el índice "inicio" e incluye todos los elementos hasta el índice "fin" pero no incluye el elemento en el índice "fin".
si se omite el valor de inicio, se asume que es 0 (el primer elemento de la lista). Si se omite el valor de fin, se asume que es el final de la lista. 
Si se omite el valor de paso, se asume que es 1 (no se saltan elementos).
En este caso, se está obteniendo desde el índice 0 hasta el índice 100 (exclusive), lo que significa que se obtendrán los primeros 100 elementos de la lista tickets.


### Indexing
El indexado es el proceso de encontrar la posición de un elemento dentro de una estructura de datos, como una lista o un arreglo. 
 En Python, las listas y otros tipos de secuencias utilizan índices basados en cero, lo que significa que el primer elemento tiene un índice de 0, 
 el segundo un índice de 1, y así sucesivamente.
si consideramos secuencias de derecha a izquierda, el último elemento tiene un índice de -1, el penúltimo de -2, y así sucesivamente.



In [ ]:
#



## 9. Comparar mediciones en una tabla

Una buena práctica es no quedarse solo con impresiones individuales.

Cuando se comparan varias mediciones, una tabla permite ver con mayor claridad qué operación tomó más tiempo.


In [ ]:
resultados = [
    {
        "Operación": medicion_validos["descripcion"],
        "Tiempo (segundos)": medicion_validos["segundos"]
    },
    {
        "Operación": medicion_invalidos["descripcion"],
        "Tiempo (segundos)": medicion_invalidos["segundos"]
    },
    {
        "Operación": medicion_creacion["descripcion"],
        "Tiempo (segundos)": medicion_creacion["segundos"]
    },
    {
        "Operación": medicion_atencion["descripcion"],
        "Tiempo (segundos)": medicion_atencion["segundos"]
    },
    {
        "Operación": medicion_cierre["descripcion"],
        "Tiempo (segundos)": medicion_cierre["segundos"]
    }
]

df_resultados = pd.DataFrame(resultados)
df_resultados["Tiempo (milisegundos)"] = df_resultados["Tiempo (segundos)"] * 1000

df_resultados



## 10. Repetir mediciones para obtener un promedio

Una sola medición puede variar por factores externos:

- carga del sistema operativo;
- uso de disco;
- procesos abiertos;
- estado del intérprete de Python;
- primera ejecución de una celda.

Por eso, cuando se quiere comparar con más seriedad, conviene repetir varias veces y calcular un promedio.


In [ ]:
#


In [ ]:

promedio = df_repeticiones_validacion["segundos"].mean()
minimo = df_repeticiones_validacion["segundos"].min()
maximo = df_repeticiones_validacion["segundos"].max()

print(f"Promedio: {promedio:.8f} segundos")
print(f"Mínimo:   {minimo:.8f} segundos")
print(f"Máximo:   {maximo:.8f} segundos")



## 11. Comparar cambio de estado directo contra método decorado

Este ejemplo ayuda a explicar una diferencia importante.

No es lo mismo medir:

```python
ticket.estado = "En proceso"
```

que medir:

```python
ticket.atender()
```

La primera operación solo cambia una propiedad validada.  
La segunda operación ejecuta lógica de negocio, decorador, historial y escritura a log.

Por eso, aunque ambas terminan modificando el estado, no cuestan lo mismo.


In [ ]:
#



## 12. Interpretación didáctica

Con estos ejercicios se puede explicar que `time.perf_counter()` sirve para medir duración, no para mostrar una hora del día.

En esta aplicación, los resultados ayudan a diferenciar entre:

- operaciones simples, como validar un folio;
- creación de objetos;
- ejecución de métodos de negocio;
- métodos decorados;
- operaciones que además escriben en archivo.

Una conclusión importante para clase:

> Si medimos una función, medimos todo lo que ocurre dentro de ella.  
> Si el método validó, cambió estado, registró historial y escribió en disco, todo eso forma parte del tiempo final.

Por eso, al analizar rendimiento, primero se debe entender qué hace realmente el código.


In [ ]:

print("Resumen final")
print("-------------")
print(f"Total de tickets creados durante la sesión: {Ticket.total_tickets_creados()}")

if os.path.exists("registro_acciones.log"):
    print(f"Archivo de log generado: registro_acciones.log")
    print(f"Tamaño actual: {os.path.getsize('registro_acciones.log')} bytes")
else:
    print("No existe archivo de log en este momento.")
